# 📊 Explicación del Análisis de Correlaciones

## ¿Qué hice y por qué?

Tenías varios archivos CSV separados con diferentes tipos de datos:
- **BD_Clima**: temperatura, humedad, lluvia (clima cada hora)
- **BD_Qin**: agua que entra al sistema (caudal de entrada)
- **BD_VolTotal**: volumen total en los estanques
- **Vol_X_TK**: volumen individual de cada uno de los 89 estanques
- **calendar_social**: fechas especiales (feriados, festivales, vacaciones)

### 🎯 El Problema
Estaban todos separados y necesitábamos saber:
1. ¿Qué variables del clima afectan más al volumen de agua?
2. ¿Los feriados o vacaciones cambian el consumo?
3. ¿Hay relación entre temperatura y la demanda?

### 🔧 La Solución
**Junté todas las tablas en una sola** usando el timestamp (fecha y hora) como "pegamento".

## Paso 1: Leer los archivos originales

Primero leí cada CSV para ver qué columnas tenían:

In [ ]:
import pandas as pd
from pathlib import Path

root = Path('c:/Users/socce/Downloads/rafa/Tesis3.0-Interfaz')
raw = root / 'data' / 'raw'

# Ejemplo: Leer clima
clima = pd.read_csv(raw / 'BD_Clima2024a202509_UTC.csv')
print("📌 Columnas en Clima:", list(clima.columns))
print("📏 Filas:", len(clima))
clima.head(3)

**Resultado:**
```
Columnas: timestamp, temp, HR, mmhr
```

- `temp` = temperatura en grados
- `HR` = humedad relativa (%)
- `mmhr` = lluvia en mm/hora

In [ ]:
# Leer volumen total
voltotal = pd.read_csv(raw / 'BD_VolTotal_X_Hr_m3_UTC.csv')
print("📌 Columnas en VolTotal:", list(voltotal.columns))
voltotal.head(3)

**Resultado:**
```
Columnas: timestamp, Volumen_Total_m3
```

Este es el **volumen total de agua en metros cúbicos** en todos los estanques.

## Paso 2: Limpiar el calendario

El archivo `calendar_social` tenía **dos timestamps** (fecha local Chile y UTC):

```csv
fecha_hora_local,timestamp_utc,anio,mes,...
2023-12-31 21:00:00-03:00,2024-01-01 00:00:00+00:00,...
```

**Problema:** Esto causa confusión porque los otros archivos usan solo UTC.

**Solución:** Eliminé `fecha_hora_local` y dejé solo `timestamp` (UTC).

In [ ]:
# Esto ya lo hice, solo es para mostrar
calendar = pd.read_csv(raw / 'calendar_social_ES_COMPLETO_20240101_20250930.csv')
print("📌 Columnas calendario (LIMPIO):", list(calendar.columns[:10]))
print("✅ Ya no tiene 'fecha_hora_local', solo 'timestamp'")

## Paso 3: Juntar todas las tablas (MERGE)

Usé el **timestamp como llave** para unir:

```python
# Pseudocódigo
merged = clima + qin + voltotal + vol_tanks + calendar
         (todas unidas por timestamp)
```

### 📊 Tabla resultante (ejemplo):

| timestamp | temp | HR | mmhr | Qin | Volumen_Total_m3 | feriado | festival_vina |
|-----------|------|-------|------|-----|------------------|---------|---------------|
| 2024-01-01 00:00 | 17.2 | 65 | 0.0 | 2500 | 98988 | 1 | 0 |
| 2024-01-01 01:00 | 16.8 | 67 | 0.0 | 2400 | 91669 | 1 | 0 |
| 2024-01-01 02:00 | 16.5 | 69 | 0.0 | 2350 | 106856 | 1 | 0 |

Ahora **todo está en una sola tabla** con 15,336 filas (una por hora).

In [ ]:
# Cargar el resultado del merge que guardé
merged = pd.read_csv(root / 'outputs' / 'merged_climate_qin_volume_sample.csv')
print(f"📏 Filas en tabla unificada: {len(merged):,}")
print(f"📌 Columnas: {list(merged.columns)}")
merged.head(5)

## Paso 4: Calcular correlaciones

### ¿Qué es la correlación?

Es un número entre **-1 y +1** que indica qué tan relacionadas están dos variables:

- **+1** = Cuando una sube, la otra también sube (relación perfecta positiva)
- **-1** = Cuando una sube, la otra baja (relación perfecta negativa)
- **0** = No hay relación

### 🎯 Preguntas que responde:

1. ¿La **temperatura** afecta al **volumen** de agua?
2. ¿La **humedad** cambia el consumo?
3. ¿Los **feriados** influyen en la demanda?

In [ ]:
# Calcular correlaciones
correlations = merged.corr(method='pearson')

# Mostrar correlaciones con Volumen_Total_m3
vol_corr = correlations['Volumen_Total_m3'].abs().sort_values(ascending=False)
print("🔥 TOP correlaciones con Volumen de Agua:")
print(vol_corr)

## 📈 Resultados Principales

### Top correlaciones con **Volumen Total**:

| Variable | Correlación | Interpretación |
|----------|-------------|----------------|
| **temp** | +0.53 | ⬆️ A mayor temperatura, MÁS agua almacenada |
| **HR** (humedad) | +0.48 | ⬆️ Más humedad → más agua almacenada |
| **mean_fill_ratio** | +0.84 | ⬆️ Llenado promedio de estanques muy correlacionado |
| **Qin** | +0.22 | ⬆️ Más entrada de agua → más volumen (obvio) |
| **feriado** | +0.06 | ⚠️ Casi no afecta (baja correlación) |

### Top correlaciones con **Qin** (entrada de agua):

| Variable | Correlación | Interpretación |
|----------|-------------|----------------|
| **temporada_turistica_alta** | +0.59 | 🏖️ En verano/turismo entra más agua |
| **temp** | +0.50 | ⬆️ Más calor → más entrada de agua |
| **HR** | +0.32 | ⬆️ Humedad también influye |

### 💡 Conclusiones:

1. ✅ **Temperatura y humedad SÍ afectan** el volumen de agua
2. ✅ **Temporada turística** influye mucho en la entrada de agua (Qin)
3. ⚠️ **Feriados y festivales** tienen poco impacto directo (puede ser que su efecto sea más complejo)
4. ✅ El **llenado promedio de estanques** es un buen indicador del volumen total

## 🎨 Visualización de correlaciones

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# Gráfico de barras con las top correlaciones
top_10 = vol_corr.head(10)

fig = go.Figure(data=[
    go.Bar(
        x=top_10.values,
        y=top_10.index,
        orientation='h',
        marker=dict(
            color=top_10.values,
            colorscale='Blues',
            showscale=True,
            colorbar=dict(title="Correlación")
        )
    )
])

fig.update_layout(
    title="Top 10 Correlaciones con Volumen Total de Agua",
    xaxis_title="Correlación (Pearson)",
    yaxis_title="Variable",
    height=500
)

fig.show()

## 📁 Archivos que creé

Guardé varios archivos CSV para que puedas revisarlos:

### 1️⃣ `data/processed/calendar_social_utc.csv`
- Calendario limpio con **solo timestamp UTC**
- Sin confusión de zona horaria

### 2️⃣ `outputs/merged_climate_qin_volume_sample.csv`
- **Tabla unificada** con todas las variables juntas
- 15,336 filas (una por hora)
- Columnas: temp, HR, mmhr, Qin, Volumen_Total_m3, feriado, etc.

### 3️⃣ `outputs/correlations_all_raw.csv`
- Matriz completa de correlaciones entre **todas** las variables

### 4️⃣ `outputs/top_corr_with_volume.csv`
- Lista ordenada de correlaciones con Volumen Total

### 5️⃣ `outputs/top_corr_with_qin.csv`
- Lista ordenada de correlaciones con Qin (entrada de agua)

## 🚀 Próximos Pasos Sugeridos

Ahora que sabemos qué variables son importantes, podemos:

### 1. Crear nuevas features climáticas avanzadas
- **Olas de calor**: Días con temperatura >30°C seguidos
- **Frentes fríos**: Caída brusca de temperatura en 6 horas
- **Tormentas**: Lluvia >20mm en 6 horas
- **Interacciones**: temp × feriado, temp × hora_pico

### 2. Reentrenar el modelo con clima
- Usar temperatura, humedad, lluvia como features
- Comparar si mejora la predicción

### 3. Mejorar la interfaz
- Agregar tabs de predicción 24h y 72h con temperatura
- Mostrar pronóstico del clima en el header

---

### ❓ ¿Te quedó más claro?

Si quieres que avance con alguno de estos pasos, dime cuál prefieres:
1. Crear features climáticas avanzadas
2. Reentrenar el modelo
3. Arreglar la interfaz